# Big Data Lab: 11.11 Sale Analytics
**Run from top to bottom in Google Colab.** This notebook generates one million simulated orders, intentionally introduces data quality problems, cleans and analyzes the data with PySpark, and exports compact results for the Streamlit dashboard.

> The generated records are synthetic, not real customer orders. The million-row raw files are intentionally saved locally in the Colab runtime and should not be pushed to GitHub.


In [ ]:
%pip -q install pyspark pyarrow


## Part B — Generate the dataset

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 1111
rng = np.random.default_rng(SEED)
random.seed(SEED)

N = 1_000_000
OUT = Path("/content/sale_analytics")
RESULTS = OUT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# Create order timestamps on November 11 with peaks at midnight and 8–10 PM.
hours = np.arange(24)
weights = np.array(
    [12, 8, 5, 3, 2.5, 2, 2.5, 3.5, 5, 7, 8.5, 9,
     11, 9.5, 9, 9.2, 10.5, 12, 15, 22, 42, 55, 38, 20],
    dtype=float
)
weights /= weights.sum()

hour_values = rng.choice(hours, size=N, p=weights)
minute_values = rng.integers(0, 60, size=N)
second_values = rng.integers(0, 60, size=N)
timestamps = (
    pd.Timestamp("2026-11-11")
    + pd.to_timedelta(hour_values, unit="h")
    + pd.to_timedelta(minute_values, unit="m")
    + pd.to_timedelta(second_values, unit="s")
)

regions = np.array([
    "NCR", "CALABARZON", "Central Luzon",
    "Central Visayas", "Davao Region", "Others"
])
region_values = rng.choice(
    regions, size=N, p=[0.38, 0.22, 0.15, 0.10, 0.07, 0.08]
).astype(object)

categories = np.array([
    "Electronics", "Fashion", "Home & Living", "Beauty",
    "Groceries", "Sports", "Toys", "Accessories"
])
category_values = rng.choice(categories, size=N)

prices = np.round(
    np.clip(rng.lognormal(mean=5.8, sigma=0.75, size=N), 50, 30000), 2
)
quantities = rng.choice(
    [1, 2, 3, 4, 5], size=N, p=[0.62, 0.22, 0.09, 0.045, 0.025]
)

payments = np.array(["COD", "GCash", "Maya", "Card"])
payment_values = rng.choice(
    payments, size=N, p=[0.42, 0.32, 0.16, 0.10]
)

status_values = np.empty(N, dtype=object)
for payment in payments:
    mask = payment_values == payment
    # COD has a higher cancellation probability than other methods.
    probs = [0.76, 0.19, 0.05] if payment == "COD" else [0.88, 0.07, 0.05]
    status_values[mask] = rng.choice(
        ["Delivered", "Cancelled", "Returned"], size=int(mask.sum()), p=probs
    )

df = pd.DataFrame({
    "order_id": np.arange(1, N + 1, dtype=np.int64),
    "timestamp": timestamps,
    "region": region_values,
    "category": category_values,
    "unit_price": prices,
    "quantity": quantities,
    "payment_method": payment_values,
    "status": status_values
})

# Inject about 2% missing regions.
missing_idx = rng.choice(N, size=int(N * 0.02), replace=False)
df.loc[missing_idx, "region"] = None

# Inject mixed-case region names, avoiding the missing-region rows.
remaining = np.setdiff1d(np.arange(N), missing_idx)
mixed_idx = rng.choice(remaining, size=int(N * 0.02), replace=False)
half = len(mixed_idx) // 2
lower_idx = mixed_idx[:half]
capital_idx = mixed_idx[half:]
df.loc[lower_idx, "region"] = [
    str(value).lower() for value in df.loc[lower_idx, "region"].to_numpy()
]
df.loc[capital_idx, "region"] = [
    str(value).capitalize() for value in df.loc[capital_idx, "region"].to_numpy()
]

# Inject duplicate order IDs (about 1% of rows).
dup_idx = rng.choice(np.arange(1, N), size=int(N * 0.01), replace=False)
df.loc[dup_idx, "order_id"] = df.loc[dup_idx - 1, "order_id"].to_numpy()

# Inject negative quantities (about 0.2% of rows).
neg_idx = rng.choice(N, size=int(N * 0.002), replace=False)
df.loc[neg_idx, "quantity"] = -df.loc[neg_idx, "quantity"].abs()

# IMPORTANT: make the entire price column strings before Parquet export.
# This avoids PyArrow errors caused by mixing floats and currency strings
# in one pandas object column.
df["unit_price"] = df["unit_price"].map(lambda value: str(value))
price_idx = rng.choice(N, size=int(N * 0.02), replace=False)
for idx in price_idx:
    numeric_price = float(prices[idx])
    df.at[idx, "unit_price"] = "₱" + format(numeric_price, ",.2f")

csv_path = OUT / "orders_raw.csv"
parquet_path = OUT / "orders_raw.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False, engine="pyarrow")

csv_size = os.path.getsize(csv_path)
parquet_size = os.path.getsize(parquet_path)

print(f"Rows generated: {len(df):,}")
print(f"CSV size: {csv_size / (1024**2):.2f} MB")
print(f"Parquet size: {parquet_size / (1024**2):.2f} MB")
print(f"CSV saved to: {csv_path}")
print(f"Parquet saved to: {parquet_path}")


**File-size record:** Copy the actual CSV and Parquet sizes printed above into this cell's notes or your report. Compression and exact sizes can vary.

## Part C — Clean the data with PySpark

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pathlib import Path

spark = (
    SparkSession.builder
    .appName("SaleAnalytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(csv_path))
)

print("Rows before cleaning:", raw.count())

# 1. Remove duplicate order IDs.
step1 = raw.dropDuplicates(["order_id"])
print("After removing duplicate order IDs:", step1.count())

# 2. Normalize region labels using lowercase keys so all capitalization variants match.
region_key = F.lower(F.trim(F.col("region")))
step2 = step1.withColumn(
    "region",
    F.when(region_key == "ncr", "NCR")
     .when(region_key == "calabarzon", "CALABARZON")
     .when(region_key == "central luzon", "Central Luzon")
     .when(region_key == "central visayas", "Central Visayas")
     .when(region_key == "davao region", "Davao Region")
     .when(region_key == "others", "Others")
     .otherwise(F.col("region"))
)
print("After standardizing region names:", step2.count())

# 3. Convert prices like ₱1,299.00 to numeric values.
step3 = step2.withColumn(
    "unit_price",
    F.regexp_replace(
        F.regexp_replace(F.col("unit_price").cast("string"), "₱", ""),
        ",", ""
    ).cast("double")
)
print("After converting text prices:", step3.count())

# 4. Remove zero or negative quantities.
step4 = step3.withColumn("quantity", F.col("quantity").cast("int"))
step4 = step4.filter(F.col("quantity") > 0)
print("After removing non-positive quantities:", step4.count())

# 5. Retain rows with missing regions, label them Unknown, and discard invalid prices.
clean = (
    step4.withColumn("region", F.coalesce(F.col("region"), F.lit("Unknown")))
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .filter(F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))
)
print("After assigning Unknown and validating prices:", clean.count())

clean.select(
    "order_id", "timestamp", "region", "category", "unit_price",
    "quantity", "payment_method", "status"
).show(5, truncate=False)


**Missing-region decision:** Keep these orders and label the region `Unknown`. This avoids discarding otherwise usable transactions while making the missing location explicit.

## Part D — Analyze at scale

In [ ]:
# 1. Delivered revenue by category
category_revenue = (clean.filter(F.col("status") == "Delivered")
    .withColumn("revenue", F.col("unit_price") * F.col("quantity"))
    .groupBy("category").agg(F.round(F.sum("revenue"), 2).alias("revenue"))
    .orderBy(F.desc("revenue")))
category_revenue.show(truncate=False)

# 2. Orders per hour
orders_per_hour = (clean.withColumn("hour", F.hour("timestamp"))
    .groupBy("hour").count().withColumnRenamed("count", "orders")
    .orderBy("hour"))
orders_per_hour.show(24)

# 3. Revenue and order count by region (revenue from delivered orders)
region_summary = (clean.groupBy("region")
    .agg(
        F.round(F.sum(F.when(F.col("status") == "Delivered", F.col("unit_price") * F.col("quantity")).otherwise(0)), 2).alias("revenue"),
        F.count("*").alias("order_count")
    ).orderBy(F.desc("revenue")))
region_summary.show(truncate=False)

# 4. Cancellation rate by payment method
cancellation = (clean.groupBy("payment_method")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("status") == "Cancelled", 1).otherwise(0)).alias("cancelled_orders")
    )
    .withColumn("cancellation_rate", F.round(F.col("cancelled_orders") / F.col("total_orders"), 4))
    .orderBy(F.desc("cancellation_rate")))
cancellation.show(truncate=False)

# 5. Average order value across delivered orders
delivered = clean.filter(F.col("status") == "Delivered").withColumn("revenue", F.col("unit_price") * F.col("quantity"))
aov_row = delivered.agg(F.round(F.sum("revenue") / F.countDistinct("order_id"), 2).alias("average_order_value")).first()
total_revenue = delivered.agg(F.round(F.sum("revenue"), 2).alias("total_revenue")).first()["total_revenue"]
total_orders = clean.count()
cancelled_rate = clean.agg(F.avg(F.when(F.col("status") == "Cancelled", 1.0).otherwise(0.0)).alias("rate")).first()["rate"]
print("Total delivered revenue:", total_revenue)
print("Total orders after cleaning:", total_orders)
print("Overall cancellation rate:", round(cancelled_rate * 100, 2), "%")
print("Average order value:", aov_row["average_order_value"])


### Compare PySpark and pandas timing for query 1

In [ ]:
# Re-run the same delivered-revenue-by-category query in PySpark and pandas.
t0 = time.perf_counter()
spark_result = (clean.filter(F.col("status") == "Delivered")
    .withColumn("revenue", F.col("unit_price") * F.col("quantity"))
    .groupBy("category").agg(F.round(F.sum("revenue"), 2).alias("revenue"))
    .collect())
spark_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
pdf = pd.read_csv(csv_path)
pdf["unit_price"] = (pdf["unit_price"].astype(str).str.replace("₱", "", regex=False)
                     .str.replace(",", "", regex=False))
pdf["unit_price"] = pd.to_numeric(pdf["unit_price"], errors="coerce")
pdf["quantity"] = pd.to_numeric(pdf["quantity"], errors="coerce")
pdf = pdf.drop_duplicates("order_id")
pdf = pdf[pdf["quantity"] > 0]
pdf_delivered = pdf[(pdf["status"] == "Delivered") & pdf["unit_price"].notna()]
pdf_delivered = pdf_delivered.copy()
pdf_delivered["revenue"] = pdf_delivered["unit_price"] * pdf_delivered["quantity"]
pandas_result = pdf_delivered.groupby("category", as_index=False)["revenue"].sum()
pandas_seconds = time.perf_counter() - t0

print(f"PySpark time: {spark_seconds:.3f} seconds")
print(f"pandas time: {pandas_seconds:.3f} seconds")
print("Usually, pandas may be faster for this single-machine one-million-row task because Spark has session and distributed-processing overhead.")
print("Spark tends to become more useful as data exceeds memory or needs distributed processing; the exact crossover depends on hardware, transformations, and cluster setup.")


### Export compact summaries

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
category_revenue.toPandas().to_csv(RESULTS / "category_revenue.csv", index=False)
orders_per_hour.toPandas().to_csv(RESULTS / "orders_per_hour.csv", index=False)
region_summary.toPandas().to_csv(RESULTS / "region_summary.csv", index=False)
cancellation.select("payment_method", "cancellation_rate").toPandas().to_csv(RESULTS / "cancellation_by_payment.csv", index=False)

summary = pd.DataFrame([{
    "total_revenue": float(total_revenue or 0),
    "total_orders": int(total_orders),
    "cancellation_rate": float(cancelled_rate or 0),
    "average_order_value": float(aov_row["average_order_value"] or 0)
}])
summary.to_csv(RESULTS / "key_metrics.csv", index=False)
print("Exported summary CSVs to:", RESULTS)
print("Remember to copy the results/ folder into your GitHub repository. Do not upload orders_raw.csv or orders_raw.parquet.")


## Finish

In [ ]:
spark.stop()
